# raw_writer

Write raw API response JSON pages to DBFS under the standard partition path.
Files are readable by `spark.read.json()` in downstream Bronze ingestion notebooks.

Dependencies:
- `%run ../config/settings`

In [ ]:
# %run ../config/settings

In [ ]:
import json as _json


def write_raw_json(
    entity:         str,
    ingestion_date: str,
    run_id:         str,
    page_idx:       int,
    payload:        dict,
) -> str:
    """
    Write one page of a raw Spotify API response to DBFS.

    Args:
        entity:         Logical entity name (e.g. 'play_history', 'tracks').
        ingestion_date: ISO date string used as partition key (e.g. '2024-01-15').
        run_id:         Unique pipeline run identifier.
        page_idx:       Zero-based page number for multi-page responses.
        payload:        The full API response dict to serialize.

    Returns:
        The DBFS path of the written file.
    """
    base      = raw_base_path(entity, ingestion_date, run_id)
    file_path = f"{base}/{entity}_page_{page_idx:04d}.json"
    dbutils.fs.put(
        file_path,
        _json.dumps(payload, ensure_ascii=False),
        overwrite=True,
    )
    return file_path


def list_raw_files(entity: str, ingestion_date: str, run_id: str) -> list[str]:
    """Return all raw JSON file paths for a given entity/run."""
    base = raw_base_path(entity, ingestion_date, run_id)
    try:
        return [f.path for f in dbutils.fs.ls(base) if f.name.endswith(".json")]
    except Exception:
        return []